# Anotador TDAH · Flujo A · Catálogo completo (1 llamada por nota)

La nota y **el catálogo completo de 63 ítems** van juntos en una sola llamada; el modelo devuelve la lista de ítems detectados con su evidencia. Es el flujo más sencillo (1 llamada por nota) y el de contexto más grande: el modelo debe considerar los 63 ítems a la vez.

*Riesgo a vigilar:* con tanto catálogo en contexto, puede pasar por alto ítems (falsos negativos) o citar de memoria (evidencias no literales).

**Común a los tres flujos (A/B/C):** el LLM solo produce ítems `{id, evidencia, justificacion}`. Las escalas afectadas y el nivel de alerta **no los decide el modelo**: se derivan de forma determinista con las reglas del instrumento. La evidencia se verifica automáticamente contra la nota.

## 1 · Parámetros

1. Imports
2. **Parámetros fijos** — paciente, notas, lista de **versiones** (modelo×T)
3. **Infra y código** — BD, Ollama y el `codigo` de la(s) versión(es) elegida(s)

`VERSIONES` = lista de (modelo, temperatura).
`VERSION_ACTIVA` = qué lanzar: `0`, `[0, 2]` o `"todas"`.
`ETIQUETA` = texto corto para el código (`"v2"`), no la lista.

In [1]:
import datetime as dt
import json
import sqlite3
import time
import unicodedata

import pandas as pd

### 1a · Parámetros 

Elige qué versiones lanzar con `VERSION_ACTIVA`.

In [ ]:
# --- Campaña ---
ETIQUETA = "v2"   # texto en el codigo de la versión de este notebook
FLUJO = "A"

# --- Caso clínico ---
PACIENTE = "PAC001"
IDS_NOTAS = None
MAX_NOTAS = 4
REPETICIONES = 3

# --- Versiones = combos modelo × temperatura ---
MODELOS = ["gemma4:e4b", "gemma4:31b"]
TEMPERATURAS = [0.2, 0.7]
VERSIONES = [(m, t) for m in MODELOS for t in TEMPERATURAS]

# Qué lanzar: 0 | [0, 2] | "todas"
VERSION_ACTIVA = 0


def resolver_versiones(seleccion=VERSION_ACTIVA, versiones=None):
    """Devuelve [(fila, modelo, temperatura), ...] a lanzar."""
    versiones = VERSIONES if versiones is None else versiones
    if seleccion == "todas" or seleccion is None:
        idxs = list(range(len(versiones)))
    elif isinstance(seleccion, int):
        idxs = [seleccion]
    else:
        idxs = list(seleccion)
    for i in idxs:
        if not 0 <= i < len(versiones):
            raise IndexError(f"Índice {i} fuera de 0..{len(versiones)-1}")
    return [(i, versiones[i][0], versiones[i][1]) for i in idxs]


EJECUTAR = resolver_versiones()
FILAS_ACTIVAS = [i for i, _, _ in EJECUTAR]

print(f"ETIQUETA={ETIQUETA!r}  flujo={FLUJO}  paciente={PACIENTE}")
print(f"Notas: IDS_NOTAS={IDS_NOTAS}  MAX_NOTAS={MAX_NOTAS}  reps={REPETICIONES}")
print(f"VERSION_ACTIVA={VERSION_ACTIVA!r} → filas {FILAS_ACTIVAS}")
print()
print("fila | modelo        | temperatura | estado")
print("-----+---------------+-------------+--------")
for i, (m, t) in enumerate(VERSIONES):
    estado = "ACTIVA" if i in FILAS_ACTIVAS else ""
    print(f"  {i}  | {m:<13} | t={t:<9} | {estado}")


### 1b · Infra y código de experimento

Solo se muestran las versiones **activas** y su código
(`flujo-etiqueta-paciente-modelo-tTEMP-fecha`).

In [ ]:
RUTA_BD = "datos/anotador.db"
OLLAMA_URL = "http://127.0.0.1:11002"
INSTRUMENTO = "instrumentos/brief2.json"
BACKEND = "langchain"


def codigo_experimento(flujo, modelo, temperatura, paciente, etiqueta, fecha=None):
    """Código de UNA versión (modelo×T)."""
    dia = fecha or dt.date.today()
    modelo_slug = str(modelo).replace(":", "_").replace("/", "_")
    return (
        f"flujo{flujo}-{etiqueta}-{paciente}-"
        f"{modelo_slug}-t{temperatura}-{dia:%Y%m%d}"
    )


EXPERIMENTOS = [
    {
        "idx": i,
        "modelo": m,
        "temperatura": t,
        "codigo": codigo_experimento(FLUJO, m, t, PACIENTE, ETIQUETA),
    }
    for i, m, t in EJECUTAR
]

MODELO = EXPERIMENTOS[0]["modelo"]
TEMPERATURA = EXPERIMENTOS[0]["temperatura"]
EXPERIMENTO = EXPERIMENTOS[0]["codigo"]
CODIGOS = [e["codigo"] for e in EXPERIMENTOS]

print(f"Versiones a lanzar ({len(EXPERIMENTOS)}):")
for e in EXPERIMENTOS:
    print(f"  [{e['idx']}] {e['modelo']}  t={e['temperatura']}")
    print(f"       → {e['codigo']}")
print(f"Infra: BD={RUTA_BD}  Ollama={OLLAMA_URL}  backend={BACKEND}")


## 2 · Datos: las notas del paciente

In [5]:
instrumento = json.load(open(INSTRUMENTO, encoding="utf-8"))
print(f"Instrumento: {instrumento['nombre']} ({len(instrumento['items'])} ítems)")

con = sqlite3.connect(RUTA_BD)

notas = pd.read_sql(
    '''
    SELECT e.id_entrada, e.id_paciente, c.rol AS informante, e.fecha,
           p.fecha_nacimiento, p.sexo, e.texto
    FROM entrada e
    JOIN paciente p USING (id_paciente)
    JOIN cuidador c ON c.id_cuidador = e.id_cuidador
    WHERE e.id_paciente = ?
    ORDER BY e.fecha
    ''',
    con, params=[PACIENTE],
)
if IDS_NOTAS:
    notas = notas[notas["id_entrada"].isin(IDS_NOTAS)]
if MAX_NOTAS:
    notas = notas.head(MAX_NOTAS)


def calcular_edad(nacimiento, observacion):
    nac = pd.to_datetime(nacimiento).date()
    obs = pd.to_datetime(observacion).date()
    return obs.year - nac.year - ((obs.month, obs.day) < (nac.month, nac.day))


notas["edad"] = [
    calcular_edad(n, f) for n, f in zip(notas["fecha_nacimiento"], notas["fecha"])
]

print(f"{PACIENTE}: {len(notas)} notas seleccionadas")
notas[["id_entrada", "informante", "fecha", "edad", "texto"]].head()

Instrumento: BRIEF-2 Familia (63 ítems)
PAC001: 4 notas seleccionadas


,id_entrada,informante,fecha,edad,texto
0,10000,madre,2024-01-08T21:40,7,Hoy imposible desayunar. Marco no aguantaba se...
1,10001,madre,2024-01-10T14:05,7,Me acaba de escribir la tutora. En el recreo M...
2,10002,madre,2024-01-12T20:55,7,Otra tarde de batalla con las tareas. Le he di...
3,10003,madre,2024-01-16T17:30,7,Hoy me ha llamado la seño de plástica. Marco l...


## 3 · Esquema de salida y derivación determinista

El contrato Pydantic (solo ítems con evidencia) y la derivación de escalas y nivel desde `reglas_coherencia`.

In [ ]:
from typing import List
from pydantic import BaseModel, Field, ConfigDict

# extra="ignore": si el LLM mete campos de más, se ignoran (no tumba toda la anotación).
# ge/le y min_length siguen validando id y textos.

class ItemDetectado(BaseModel):
    model_config = ConfigDict(extra="ignore")

    id: int = Field(..., ge=1, le=63,
        description="Número del ítem BRIEF-2 detectado (1-63).")
    evidencia: str = Field(..., min_length=3,
        description="Cita LITERAL de la nota, sin parafrasear.")
    justificacion: str = Field(..., min_length=5,
        description="Por qué esa cita corresponde a ese ítem concreto.")


class ListaItems(BaseModel):
    model_config = ConfigDict(extra="ignore")
    items: List[ItemDetectado] = Field(
        default_factory=list,
        description="Todos los ítems detectados. Lista vacía si ninguno.",
    )


In [11]:
print(ListaItems.model_json_schema())

{'$defs': {'ItemDetectado': {'additionalProperties': False, 'properties': {'id': {'description': 'Número del ítem BRIEF-2 detectado (1-63).', 'maximum': 63, 'minimum': 1, 'title': 'Id', 'type': 'integer'}, 'evidencia': {'description': 'Cita LITERAL de la nota, sin parafrasear.', 'minLength': 5, 'title': 'Evidencia', 'type': 'string'}, 'justificacion': {'description': 'Por qué esa cita corresponde a ese ítem concreto.', 'minLength': 10, 'title': 'Justificacion', 'type': 'string'}}, 'required': ['id', 'evidencia', 'justificacion'], 'title': 'ItemDetectado', 'type': 'object'}}, 'additionalProperties': False, 'properties': {'items': {'description': 'Todos los ítems detectados. Lista vacía si ninguno.', 'items': {'$ref': '#/$defs/ItemDetectado'}, 'title': 'Items', 'type': 'array'}}, 'title': 'ListaItems', 'type': 'object'}


In [9]:
# El LLM solo produce ítems. Escalas y nivel de alerta se DERIVAN de forma
# determinista: menos cosas que puede alucinar el modelo, más auditable.
ESCALA_DE_ITEM = {it["id"]: it["escala"] for it in instrumento["items"]}
REGLAS = instrumento["reglas_coherencia"]


def derivar_escalas_y_nivel(ids_items):
    '''Escalas afectadas y nivel de alerta a partir de los ítems detectados.

    Reglas del instrumento (reglas_coherencia de brief2.json):
    alto si nº ítems >= alerta_alto_min_items; moderado si >= alerta_moderado_min_items.
    '''
    unicos = set(ids_items)
    escalas = sorted({ESCALA_DE_ITEM[i] for i in unicos if i in ESCALA_DE_ITEM})
    if len(unicos) >= REGLAS["alerta_alto_min_items"]:
        nivel = "alto"
    elif len(unicos) >= REGLAS["alerta_moderado_min_items"]:
        nivel = "moderado"
    else:
        nivel = "bajo"
    return escalas, nivel


print(derivar_escalas_y_nivel([1, 30, 10, 6]))   # -> 4 ítems = alto
print(derivar_escalas_y_nivel([3]))              # -> 1 ítem  = bajo

(['control_emocional', 'inhibicion'], 'alto')
(['memoria_trabajo'], 'bajo')


## 4 · Prompts

In [10]:
COMILLAS = '"' * 3


def construir_prompt_sistema(instrumento):
    catalogo = "\n".join(
        f"  {it['id']}: [{it['escala']}] {it['texto']}" for it in instrumento["items"]
    )
    return f'''Eres un {instrumento["rol_anotador"]}.

Analiza la nota libre de un cuidador sobre su hijo/a e identifica qué ítems del
instrumento {instrumento["nombre"]} se observan en ella.

## CATÁLOGO DE ÍTEMS ({len(instrumento["items"])} ítems)
{catalogo}

## INSTRUCCIONES
Para CADA ítem que detectes aporta:
- "id": el número del ítem del catálogo.
- "evidencia": la cita TEXTUAL de la nota que lo sustenta, copiada literalmente.
  Si no puedes citar un fragmento literal, NO incluyas el ítem.
- "justificacion": una frase explicando por qué esa evidencia corresponde al ítem.

Incluye solo ítems observables en la nota. Si la nota no refleja ningún ítem,
devuelve la lista vacía.'''


def construir_prompt_usuario(e):
    return f'''## CONTEXTO DEL PACIENTE
- Edad: {e.edad} años
- Sexo: {e.sexo}
- Informante: {e.informante}
- Fecha de la nota: {e.fecha}

## NOTA DEL CUIDADOR
{COMILLAS}{e.texto}{COMILLAS}

Identifica los ítems observables con su evidencia.'''


prompt_sistema = construir_prompt_sistema(instrumento)
print(prompt_sistema[:400] + "\n[...]")

Eres un psicólogo clínico infantil especializado en TDAH y en el instrumento BRIEF-2.

Analiza la nota libre de un cuidador sobre su hijo/a e identifica qué ítems del
instrumento BRIEF-2 Familia se observan en ella.

## CATÁLOGO DE ÍTEMS (63 ítems)
  1: [inhibicion] Es inquieto o inquieta.
  2: [flexibilidad] Se resiste o le cuesta aceptar maneras alternativas de resolver un problema.
  3: [memori
[...]


## 5 · El modelo y la función de anotación del flujo

Ejecutar **después** del esquema Pydantic (§3) y de los prompts (§4).

### Política de formato y reintentos (fijada *ex ante*)

| Parámetro | Valor | Motivo |
|-----------|-------|--------|
| `num_predict` | 4096 | Holgado: evitar truncar JSON a mitad (`done_reason=length`) |
| `format` | `"json"` | Constraint de decodificación en Ollama |
| `include_raw` | `True` | Nunca perder el texto ni `done_reason` / `eval_count` |
| `MAX_INTENTOS_FORMATO` | 3 | Se registran **todos** los intentos; se usa el **primero válido** |

Un intento con `formato_ok=0` es **fallo técnico** (parseo/truncamiento), **no** `items=[]`.
No entra en el denominador de sensibilidad/detección; se reporta aparte como tasa de fallo de formato.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

# --- Política ex ante (no cambiar a mitad de una corrida publicada) ---
NUM_PREDICT = 4096          # holgado frente a truncamiento JSON
MAX_INTENTOS_FORMATO = 3    # máx. intentos por nota; se usa el primero válido
N_LLAMADAS_POR_NOTA = MAX_INTENTOS_FORMATO  # cota superior de llamadas/nota


def configurar_modelo(modelo, temperatura):
    """LLM estructurado: JSON forzado + raw para auditoría (done_reason, eval_count)."""
    llm = ChatOllama(
        model=modelo,
        base_url=OLLAMA_URL,
        temperature=temperatura,
        num_predict=NUM_PREDICT,
        format="json",
    )
    return llm.with_structured_output(
        ListaItems, method="json_schema", include_raw=True
    )


def _item_a_dict(it):
    if isinstance(it, dict):
        return it
    if hasattr(it, "model_dump"):
        return it.model_dump()
    return {
        "id": getattr(it, "id", None),
        "evidencia": getattr(it, "evidencia", ""),
        "justificacion": getattr(it, "justificacion", ""),
    }


def _meta_desde_raw(raw):
    """Extrae done_reason / eval_count de la respuesta Ollama (vía LangChain)."""
    md = getattr(raw, "response_metadata", None) or {}
    usage = getattr(raw, "usage_metadata", None) or {}
    content = getattr(raw, "content", None)
    if isinstance(content, list):
        content = "".join(
            (b.get("text", "") if isinstance(b, dict) else str(b)) for b in content
        )
    return {
        "respuesta_cruda": content if isinstance(content, str) else (
            str(content) if content is not None else None
        ),
        "done_reason": md.get("done_reason"),
        "eval_count": md.get("eval_count") or usage.get("output_tokens"),
        "prompt_eval_count": md.get("prompt_eval_count") or usage.get("input_tokens"),
        "total_duration_ns": md.get("total_duration"),
    }


def _intento_anotar(motor, e):
    """Una sola llamada con include_raw → dict de intento."""
    t0 = time.time()
    registro = {
        "ok": False,
        "items": [],
        "parsed": None,
        "parsing_error": None,
        "respuesta_cruda": None,
        "done_reason": None,
        "eval_count": None,
        "prompt_eval_count": None,
        "latencia_s": None,
        "error": None,
        "truncado": False,
    }
    try:
        out = motor.invoke([
            SystemMessage(prompt_sistema),
            HumanMessage(construir_prompt_usuario(e)),
        ])
    except Exception as exc:
        registro["error"] = f"{type(exc).__name__}: {exc}"
        registro["latencia_s"] = round(time.time() - t0, 3)
        return registro

    registro["latencia_s"] = round(time.time() - t0, 3)

    if not isinstance(out, dict):
        datos = out if isinstance(out, dict) else out.model_dump()
        registro["ok"] = True
        registro["items"] = [_item_a_dict(it) for it in (datos.get("items") or [])]
        registro["parsed"] = datos
        return registro

    raw = out.get("raw")
    parsed = out.get("parsed")
    perr = out.get("parsing_error")
    registro.update(_meta_desde_raw(raw))
    registro["truncado"] = registro.get("done_reason") == "length"
    registro["parsing_error"] = str(perr) if perr else None

    if parsed is not None and perr is None:
        datos = parsed if isinstance(parsed, dict) else parsed.model_dump()
        registro["ok"] = True
        registro["parsed"] = datos
        registro["items"] = [_item_a_dict(it) for it in (datos.get("items") or [])]
    else:
        registro["ok"] = False
        if registro["truncado"] and not registro["parsing_error"]:
            registro["parsing_error"] = "truncado: done_reason=length"
    return registro


def anotar_nota(e, llm_out=None, max_intentos=None):
    """Anota una nota con hasta MAX_INTENTOS_FORMATO intentos.

    Política: se registran TODOS los intentos; se usa el PRIMERO válido.
    Retorna (items, formato_ok, traza) donde traza incluye auditoría completa.
    formato_ok=False ⇒ fallo técnico (no confundir con items=[] válido).
    """
    motor = llm_out if llm_out is not None else llm_estructurado
    max_intentos = MAX_INTENTOS_FORMATO if max_intentos is None else max_intentos
    intentos = []
    elegido = None

    for i in range(1, max_intentos + 1):
        reg = _intento_anotar(motor, e)
        reg["intento"] = i
        intentos.append(reg)
        if reg["ok"]:
            elegido = reg
            break

    if elegido is not None:
        traza = {
            "formato_ok": True,
            "fallo_tecnico": False,
            "intento_usado": elegido["intento"],
            "n_intentos": len(intentos),
            "items": elegido["items"],
            "parsed": elegido["parsed"],
            "respuesta_cruda": elegido["respuesta_cruda"],
            "done_reason": elegido["done_reason"],
            "eval_count": elegido["eval_count"],
            "prompt_eval_count": elegido["prompt_eval_count"],
            "truncado": elegido["truncado"],
            "intentos": intentos,
        }
        return elegido["items"], True, traza

    ultimo = intentos[-1]
    traza = {
        "formato_ok": False,
        "fallo_tecnico": True,
        "intento_usado": None,
        "n_intentos": len(intentos),
        "items": [],
        "parsed": None,
        "respuesta_cruda": ultimo.get("respuesta_cruda"),
        "done_reason": ultimo.get("done_reason"),
        "eval_count": ultimo.get("eval_count"),
        "prompt_eval_count": ultimo.get("prompt_eval_count"),
        "truncado": any(r.get("truncado") for r in intentos),
        "parsing_error": ultimo.get("parsing_error") or ultimo.get("error"),
        "intentos": intentos,
    }
    return [], False, traza


llm_estructurado = configurar_modelo(MODELO, TEMPERATURA)
print(f"LLM listo: {MODELO}  T={TEMPERATURA}")
print(f"num_predict={NUM_PREDICT}  format=json  max_intentos={MAX_INTENTOS_FORMATO}")
print("anotar_nota() con include_raw + reintentos definida.")


In [ ]:
# Celda reservada (la lógica vive en §5).
# Re-ejecuta §5 si cambias ListaItems / NUM_PREDICT / MAX_INTENTOS_FORMATO.
print(f"Política activa: num_predict={NUM_PREDICT} · intentos≤{MAX_INTENTOS_FORMATO} · format=json")


## 6 · Verificación automática de la evidencia

In [7]:
def _normalizar(s):
    '''Minúsculas, sin tildes y espacios colapsados.'''
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return " ".join(s.split())


def verificar_evidencias(items, texto_nota):
    '''Fracción de ítems cuya evidencia aparece literalmente en la nota.

    Devuelve (fraccion, detalle) con detalle = [(id_item, True/False), ...].
    (None, []) si no hay ítems.
    '''
    if not items:
        return None, []
    t = _normalizar(texto_nota)
    detalle = [
        (it.get("id"), _normalizar(it.get("evidencia", "")) in t) for it in items
    ]
    return sum(ok for _, ok in detalle) / len(detalle), detalle

## 7 · Una nota de ejemplo

In [ ]:
# Comprobaciones mínimas (si falla, falta ejecutar celdas anteriores)
for _nombre in ("notas", "MODELO", "TEMPERATURA", "prompt_sistema",
                "ListaItems", "configurar_modelo", "anotar_nota", "verificar_evidencias",
                "derivar_escalas_y_nivel", "MAX_INTENTOS_FORMATO", "NUM_PREDICT"):
    if _nombre not in globals():
        raise NameError(
            f"Falta '{_nombre}'. Ejecuta en orden §1 → §2 → §3 → §4 → §5 → §6 → §7."
        )

llm_estructurado = configurar_modelo(MODELO, TEMPERATURA)

ejemplo = notas.iloc[0]
print(f"Nota {ejemplo.id_entrada} · {ejemplo.informante} · {ejemplo.fecha}")
print(f"Modelo: {MODELO}  T={TEMPERATURA}  num_predict={NUM_PREDICT}")
print(f"Texto: {ejemplo.texto[:250]}\n")

t0 = time.time()
items, formato_ok, traza = anotar_nota(ejemplo)
print(f"Latencia total: {time.time() - t0:.1f}s · formato_ok={formato_ok} "
      f"· intentos={traza['n_intentos']} · usado={traza['intento_usado']}")
print(f"done_reason={traza.get('done_reason')!r}  eval_count={traza.get('eval_count')}  "
      f"truncado={traza.get('truncado')}\n")

for reg in traza["intentos"]:
    print(f"  intento {reg['intento']}: ok={reg['ok']}  done_reason={reg.get('done_reason')!r}  "
          f"eval_count={reg.get('eval_count')}  lat={reg.get('latencia_s')}s  "
          f"err={reg.get('parsing_error') or reg.get('error')}")

if not formato_ok:
    print("\nFALLO TÉCNICO (no es items=[]):")
    print(f"  parsing_error: {traza.get('parsing_error')}")
    cruda = traza.get("respuesta_cruda") or ""
    print(f"  cruda (últimos 400 chars): ...{cruda[-400:]}")
elif not items:
    print("El modelo devolvió items=[] válido (detección negativa, no fallo técnico).")

fraccion, detalle = verificar_evidencias(items, ejemplo.texto)
verificado = dict(detalle)
escalas, nivel = derivar_escalas_y_nivel([it["id"] for it in items])

print(f"\nÍtems detectados : {sorted(it['id'] for it in items)}")
print(f"Escalas (derivadas): {escalas}")
print(f"Nivel (derivado)   : {nivel}")
if fraccion is not None:
    print(f"Evidencia verificada: {fraccion:.0%}")
print()
for it in items:
    marca = "OK " if verificado.get(it["id"]) else "?? "
    print(f"  [{marca}] ítem {it['id']}: \"{it['evidencia']}\"")
    print(f"        → {it['justificacion']}")


## 8 · El experimento

Cada fila de `experimento` es la anotación COMPLETA de una nota en una repetición.
El `codigo` incluye flujo, versión, paciente, modelo, temperatura y fecha
(`flujoA-v2-PAC001-gemma4_31b-t0.7-20260805`).

### Fallo técnico vs detección negativa

| `formato_ok` | Significado | ¿Entra en sensibilidad? |
|--------------|-------------|-------------------------|
| `1` | JSON válido (aunque `items=[]`) | **Sí** — es observación de detección |
| `0` | Fallo técnico (parseo / `done_reason=length` / error) | **No** — excluir del denominador |

Tasa de fallo de formato = `mean(formato_ok==0)` → resultado de viabilidad del stack local, no FN.

> Relanzar con el mismo código añade filas. Para limpiar:
> `python3 datos/limpiar_experimentos.py --codigo '...'`
> o `python3 datos/limpiar_experimentos.py --flujo A`


In [ ]:
con.execute("""
CREATE TABLE IF NOT EXISTS experimento (
    id                INTEGER PRIMARY KEY,
    codigo            TEXT NOT NULL,
    creada_en         TEXT NOT NULL,
    backend           TEXT NOT NULL,
    modelo            TEXT NOT NULL,
    temperatura       REAL NOT NULL,
    semana            INTEGER,
    id_paciente       TEXT,
    id_entrada        INTEGER,
    repeticion        INTEGER,
    formato_ok        INTEGER,
    items_detectados  TEXT,
    escalas_afectadas TEXT,
    nivel_alerta      TEXT,
    nota_clinica      TEXT,
    justificacion     TEXT,
    latencia_s        REAL,
    respuesta_cruda   TEXT,
    items_detalle     TEXT,
    evidencia_ok      REAL
)""")
for col, tipo in [
    ("respuesta_cruda", "TEXT"),
    ("items_detalle", "TEXT"),
    ("evidencia_ok", "REAL"),
    ("n_intentos", "INTEGER"),
    ("intento_usado", "INTEGER"),
    ("done_reason", "TEXT"),
    ("eval_count", "INTEGER"),
    ("intentos_detalle", "TEXT"),
    ("fallo_tecnico", "INTEGER"),
]:
    try:
        con.execute(f"ALTER TABLE experimento ADD COLUMN {col} {tipo}")
    except sqlite3.OperationalError:
        pass
con.commit()

por_corrida = len(notas) * REPETICIONES
total_anot = por_corrida * len(EXPERIMENTOS)
total_llamadas_max = total_anot * MAX_INTENTOS_FORMATO
print(f"{len(EXPERIMENTOS)} ejecucion(s) · {len(notas)} notas × {REPETICIONES} reps "
      f"= {total_anot} anotaciones · ≤{total_llamadas_max} llamadas "
      f"(máx {MAX_INTENTOS_FORMATO} intentos/nota, num_predict={NUM_PREDICT})\n")

hechas = 0
for exp in EXPERIMENTOS:
    modelo = exp["modelo"]
    temperatura = exp["temperatura"]
    codigo = exp["codigo"]
    llm_out = configurar_modelo(modelo, temperatura)
    print(f"=== [{exp['idx']}] {codigo} ===")

    for _, e in notas.iterrows():
        for rep in range(REPETICIONES):
            t0 = time.time()
            items, formato_ok, traza = anotar_nota(e, llm_out)
            latencia = time.time() - t0
            fallo_tecnico = int(not formato_ok)
            # Evidencia solo tiene sentido si el formato es válido
            if formato_ok:
                fraccion, _ = verificar_evidencias(items, e.texto)
                escalas, nivel = derivar_escalas_y_nivel([it["id"] for it in items])
            else:
                fraccion, escalas, nivel = None, [], None

            con.execute(
                "INSERT INTO experimento (codigo, creada_en, backend, modelo, temperatura, "
                "semana, id_paciente, id_entrada, repeticion, formato_ok, items_detectados, "
                "escalas_afectadas, nivel_alerta, nota_clinica, justificacion, latencia_s, "
                "respuesta_cruda, items_detalle, evidencia_ok, "
                "n_intentos, intento_usado, done_reason, eval_count, intentos_detalle, fallo_tecnico) "
                "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                (codigo, dt.datetime.now().isoformat(timespec="seconds"), BACKEND,
                 modelo, temperatura, None, e.id_paciente, int(e.id_entrada), rep,
                 int(formato_ok),
                 json.dumps(sorted(it["id"] for it in items)) if formato_ok else None,
                 json.dumps(escalas) if formato_ok else None,
                 nivel, None, None, latencia,
                 traza.get("respuesta_cruda"),
                 json.dumps(items, ensure_ascii=False) if formato_ok else None,
                 fraccion,
                 traza.get("n_intentos"),
                 traza.get("intento_usado"),
                 traza.get("done_reason"),
                 traza.get("eval_count"),
                 json.dumps(traza.get("intentos"), ensure_ascii=False),
                 fallo_tecnico),
            )
            con.commit()
            hechas += 1
            if not formato_ok:
                marca = (f"FALLO TÉCNICO · done_reason={traza.get('done_reason')!r} "
                         f"· eval_count={traza.get('eval_count')} "
                         f"· intentos={traza['n_intentos']}")
            else:
                ev = f"evidencia {fraccion:.0%}" if fraccion is not None else "items=[]"
                marca = f"{len(items)} ítems · nivel {nivel} · {ev}"
            print(f"  [{hechas:>3}/{total_anot}] nota {e.id_entrada} rep {rep + 1} → "
                  f"{marca} ({latencia:.1f}s)")

print("\nGuardado:")
for exp in EXPERIMENTOS:
    print(f"  {exp['codigo']}")


## 8b · Comparación entre repeticiones (ítems + justificaciones)

Ejecutar **después** de la celda del experimento (§8).

Para cada nota: tabla de ítems por repetición, Jaccard por pares,
y la **evidencia + justificación** de cada ítem para comparar el razonamiento.


In [ ]:
# Comparación detallada: ítems + justificaciones por repetición
placeholders = ",".join("?" * len(CODIGOS))
df_comp = pd.read_sql(
    f"""SELECT id, codigo, id_entrada, repeticion, formato_ok,
               items_detectados, items_detalle, nivel_alerta, latencia_s,
               done_reason, eval_count, n_intentos, intento_usado, fallo_tecnico
        FROM experimento WHERE codigo IN ({placeholders})
        ORDER BY codigo, id_entrada, repeticion, id""",
    con, params=CODIGOS,
)

# Texto de la nota (una vez)
textos = {
    int(r.id_entrada): r.texto
    for _, r in notas.iterrows()
}


def _parse_items(detalle, ids_json):
    if isinstance(detalle, str) and detalle.strip():
        try:
            return json.loads(detalle)
        except json.JSONDecodeError:
            pass
    if isinstance(ids_json, str) and ids_json.strip():
        return [{"id": i, "evidencia": "", "justificacion": ""} for i in json.loads(ids_json)]
    return []


def _jaccard(a, b):
    a, b = frozenset(a), frozenset(b)
    if not (a | b):
        return 1.0
    return len(a & b) / len(a | b)


for codigo, g_cod in df_comp.groupby("codigo", sort=False):
    print("=" * 88)
    print(codigo)
    print("=" * 88)

    for id_entrada, g in g_cod.groupby("id_entrada", sort=True):
        g = g.sort_values(["repeticion", "id"])
        print(f"\n### Nota {id_entrada}")
        texto = textos.get(int(id_entrada), "")
        if texto:
            print(f"Texto: {texto[:220]}{'…' if len(texto) > 220 else ''}\n")

        filas_tabla = []
        sets = []
        reps_ok = []

        for _, r in g.iterrows():
            items = _parse_items(r["items_detalle"], r["items_detectados"])
            ids = sorted(it["id"] for it in items if it.get("id") is not None)
            ok = int(r["formato_ok"]) == 1
            if ok:
                sets.append(frozenset(ids))
                reps_ok.append(int(r["repeticion"]))
            filas_tabla.append({
                "rep": int(r["repeticion"]),
                "formato_ok": ok,
                "ítems": ids if ok else "(fallo técnico)",
                "n": len(ids) if ok else None,
                "nivel": r["nivel_alerta"],
                "lat_s": round(float(r["latencia_s"]), 1) if r["latencia_s"] is not None else None,
                "done_reason": r.get("done_reason"),
                "intentos": r.get("n_intentos"),
            })

        display(pd.DataFrame(filas_tabla))

        # Jaccard por pares (solo formato_ok)
        if len(sets) >= 2:
            print("Jaccard por pares:")
            pares, acum = 0, 0.0
            for i in range(len(sets)):
                for j in range(i + 1, len(sets)):
                    a, b = sets[i], sets[j]
                    jacc = _jaccard(a, b)
                    acum += jacc
                    pares += 1
                    print(
                        f"  rep {reps_ok[i]} vs rep {reps_ok[j]} → {jacc:.2f}  "
                        f"∩={sorted(a & b)}  "
                        f"solo_rep{reps_ok[i]}={sorted(a - b)}  "
                        f"solo_rep{reps_ok[j]}={sorted(b - a)}"
                    )
            print(f"  → Jaccard medio: {acum / pares:.2f}")
        elif len(sets) == 1:
            print("Solo 1 repetición válida: Jaccard no aplicable.")
        else:
            print("Sin repeticiones válidas (todas fallo técnico).")

        # Detalle evidencia/justificación
        print("\nJustificaciones por repetición:")
        for _, r in g.iterrows():
            rep = int(r["repeticion"])
            if int(r["formato_ok"]) != 1:
                print(f"\n  --- rep {rep} · FALLO TÉCNICO · done_reason={r.get('done_reason')!r} ---")
                continue
            items = _parse_items(r["items_detalle"], r["items_detectados"])
            print(f"\n  --- rep {rep} · ítems {sorted(it['id'] for it in items)} ---")
            if not items:
                print("      (items=[] válido)")
            for it in sorted(items, key=lambda x: x.get("id") or 0):
                print(f"      ítem {it.get('id')}:")
                print(f"        evidencia    : {it.get('evidencia', '')!r}")
                print(f"        justificación: {it.get('justificacion', '')}")

        # Ítems que oscilan (aparecen en algunas reps pero no en todas)
        if len(sets) >= 2:
            union = set.union(*[set(s) for s in sets])
            inter = set.intersection(*[set(s) for s in sets])
            oscilan = sorted(union - inter)
            estables = sorted(inter)
            print(f"\n  Estables (en todas las reps válidas): {estables}")
            print(f"  Oscilan (no en todas)             : {oscilan}")
        print()


## Resultados

Métricas de **detección / estabilidad / evidencia** solo sobre filas con `formato_ok=1`.
Las filas `formato_ok=0` (`fallo_tecnico=1`) se reportan aparte como tasa de fallo de formato
y **no** cuentan como falsos negativos.


In [ ]:
placeholders = ",".join("?" * len(CODIGOS))
df = pd.read_sql(
    f"SELECT * FROM experimento WHERE codigo IN ({placeholders})",
    con, params=CODIGOS,
)
print(f"{len(df)} anotaciones en {len(CODIGOS)} código(s)\n")
for c in CODIGOS:
    print(f"  {c}: {(df['codigo'] == c).sum()} filas")
print()

# --- Aislar fallos técnicos (no FN) ---
if "fallo_tecnico" not in df.columns:
    df["fallo_tecnico"] = (df["formato_ok"] == 0).astype(int)
else:
    df["fallo_tecnico"] = (
        df["fallo_tecnico"].fillna((df["formato_ok"] == 0).astype(int)).astype(int)
    )

df_ok = df[df["formato_ok"] == 1].copy()      # válidas para detección
df_tech = df[df["formato_ok"] == 0].copy()    # fallos técnicos

n_total = len(df)
n_tech = len(df_tech)
n_ok = len(df_ok)
tasa_fallo_formato = n_tech / n_total if n_total else float("nan")

print("=== Viabilidad de formato (todas las filas) ===")
print(f"Total anotaciones                     : {n_total}")
print(f"Fallos técnicos (formato_ok=0)        : {n_tech}  ({tasa_fallo_formato:.1%})")
print(f"Observaciones válidas (formato_ok=1)  : {n_ok}")
if n_tech and "done_reason" in df_tech.columns:
    print("done_reason en fallos:")
    print(df_tech["done_reason"].value_counts(dropna=False).to_string())
    if "eval_count" in df_tech.columns:
        print(f"eval_count en fallos (mediana)        : {df_tech['eval_count'].median()}")
print()
print("Las filas con formato_ok=0 se EXCLUYEN del denominador de detección/sensibilidad.")
print("Un items=[] con formato_ok=1 SÍ cuenta (detección negativa válida).\n")


def acuerdo_modal(valores):
    s = valores.dropna()
    return round(s.value_counts().iloc[0] / len(s), 2) if len(s) else None


def jaccard_repeticiones(grupo):
    """Jaccard medio entre pares de repeticiones (solo filas válidas)."""
    conjuntos = [frozenset(json.loads(x)) for x in grupo if isinstance(x, str)]
    if len(conjuntos) < 2:
        return None
    pares, total = 0, 0.0
    for i in range(len(conjuntos)):
        for j in range(i + 1, len(conjuntos)):
            a, b = conjuntos[i], conjuntos[j]
            total += len(a & b) / len(a | b) if (a | b) else 1.0
            pares += 1
    return round(total / pares, 2)


# Resumen de detección SOLO sobre df_ok
if len(df_ok):
    resumen = (
        df_ok.groupby(["codigo", "id_entrada"])
        .agg(
            repeticiones_validas=("repeticion", "count"),
            jaccard_items=("items_detectados", jaccard_repeticiones),
            acuerdo_nivel=("nivel_alerta", acuerdo_modal),
            evidencia_media=("evidencia_ok", "mean"),
            latencia_media=("latencia_s", "mean"),
        )
        .round(2)
    )
    por_codigo = resumen.groupby("codigo").mean(numeric_only=True).round(2)
    print("=== Detección / estabilidad (solo formato_ok=1) ===")
    display(por_codigo)
    print()
    print(f"Evidencia verificada (válidas)        : {df_ok['evidencia_ok'].mean():.0%}")
    print(f"Latencia media (válidas)              : {df_ok['latencia_s'].mean():.1f}s")
else:
    resumen = pd.DataFrame()
    print("No hay filas con formato_ok=1 para métricas de detección.")

# Tasa de fallo por código (todas las filas)
fallo_por_codigo = (
    df.groupby("codigo")
    .agg(
        n=("formato_ok", "size"),
        tasa_fallo_formato=("formato_ok", lambda s: round(1 - s.mean(), 3)),
        n_fallos=("formato_ok", lambda s: int((s == 0).sum())),
    )
)
print("\n=== Tasa de fallo de formato por código (publicable) ===")
display(fallo_por_codigo)

# Demostración sobre la corrida ya guardada (si aplica)
print("\n=== Filtro listo para sensibilidad (plantilla) ===")
print("df_deteccion = df[df['formato_ok'] == 1]   # denominador")
print("df_fallo     = df[df['formato_ok'] == 0]   # tasa de fallo, aparte")

resumen


## 9 · Comparación entre flujos

Ejecutar los tres cuadernos (A, B y C) con **el mismo paciente, las mismas notas, el mismo modelo y la misma temperatura**, y comparar los códigos de experimento en `comparacion_experimentos.ipynb`:

- **Estabilidad**: Jaccard de ítems y acuerdo del nivel entre repeticiones (**solo `formato_ok=1`**).
- **Fidelidad**: evidencia verificada (¿las citas existen?).
- **Coste**: latencia por nota (A: 1 llamada · C: 9 · B: 63).
- **Viabilidad**: tasa de fallo de formato (`formato_ok=0`), reportada aparte; no confundir con FN.

La pregunta de la ablación: ¿cuánta estructura de contexto necesita el modelo para anotar de forma estable, y a qué coste?
